<a href="https://colab.research.google.com/github/conecrew/curso-ciberwall-univ-salamanca/blob/main/ver_si_es_wordpress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

script para detectar si la web es wordpress

In [2]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

BASE_URL = "https://enriqueite.com/demoGPT/"

# Comprobaciones de rutas y recursos públicos habituales.
RUTAS = [
    "wp-admin/",
    "wp-login.php",
    "wp-content/",
    "wp-includes/",
    "wp-json/",
    "?rest_route=/",
    "wp-sitemap.xml",
    "robots.txt",
    "sitemap.xml",
    "readme.html",
    "license.txt",
    "xmlrpc.php",
    "wp-content/plugins/",
    "wp-content/themes/",
    "wp-content/uploads/",
    "wp-content/languages/",
    "wp-includes/js/",
    "wp-includes/css/",
    "wp-admin/css/",
    "wp-admin/js/",
    "wp-admin/images/",
    "wp-admin/admin-ajax.php",
    "wp-includes/images/",
    "wp-content/mu-plugins/",
]

sesion = requests.Session()
sesion.headers["User-Agent"] = (
    "CyberwallAcademy-EducationalAudit/1.0"
)

def inspeccionar_portada():
    print("=== INSPECCIÓN DE LA PORTADA ===")
    try:
        r = sesion.get(BASE_URL, timeout=12)
        print("Estado HTTP:", r.status_code)
        print("URL final:", r.url)
        print("Tipo de contenido:",
              r.headers.get("Content-Type", "No indicado"))
        print("Servidor:", r.headers.get("Server", "No indicado"))

        html = r.text
        soup = BeautifulSoup(html, "html.parser")

        generator = soup.find("meta", attrs={"name": "generator"})
        if generator:
            print("Meta generator:",
                  generator.get("content", "Sin contenido"))
        else:
            print("Meta generator: no encontrado")

        api_links = soup.find_all(
            "link", attrs={"rel": "https://api.w.org/"}
        )
        for link in api_links:
            print("Descubrimiento REST:",
                  link.get("href", "Sin URL"))

        indicadores = {
            "wp-content en HTML": "wp-content/" in html.lower(),
            "wp-includes en HTML": "wp-includes/" in html.lower(),
            "REST API indicada en cabecera":
                "api.w.org" in r.headers.get("Link", ""),
            "REST API indicada en HTML": bool(api_links),
            "wp-json mencionado en HTML": "wp-json" in html.lower(),
        }

        print("\nIndicadores:")
        for nombre, encontrado in indicadores.items():
            print(f"  {'Sí' if encontrado else 'No'} - {nombre}")

        if any(indicadores.values()):
            print("\nConclusión: hay indicios compatibles con WordPress.")
        else:
            print("\nConclusión: WordPress no confirmado.")

    except requests.RequestException as e:
        print("Error al consultar la portada:", e)


def comprobar_rutas():
    print("\n=== COMPROBACIÓN DE RUTAS ===")
    print(f"{'HTTP':<6} {'TAMAÑO':<10} {'RUTA':<38} REDIRECCIÓN")
    print("-" * 100)

    # Referencia para detectar páginas de error genéricas.
    referencia_url = urljoin(
        BASE_URL, "__ruta_inexistente_prueba_cyberwall__"
    )
    try:
        referencia = sesion.get(
            referencia_url,
            timeout=10,
            allow_redirects=False,
            stream=True
        )
        ref_status = referencia.status_code
        ref_length = referencia.headers.get("Content-Length", "?")
        referencia.close()
        print("Referencia inexistente:",
              ref_status, "| tamaño declarado:", ref_length)
    except requests.RequestException:
        ref_status = None

    for ruta in RUTAS:
        url = urljoin(BASE_URL, ruta)
        try:
            r = sesion.get(
                url,
                timeout=10,
                allow_redirects=False,
                stream=True
            )
            estado = r.status_code
            longitud = r.headers.get("Content-Length", "?")
            destino = r.headers.get("Location", "")

            nota = ""
            if estado == ref_status:
                nota = " (comparar con referencia)"

            print(
                f"{estado:<6} {longitud:<10} "
                f"{ruta:<38} {destino}{nota}"
            )
            r.close()

        except requests.RequestException as e:
            print(f"ERROR  {'?':<10} {ruta:<38} {e}")


inspeccionar_portada()
comprobar_rutas()


=== INSPECCIÓN DE LA PORTADA ===
Estado HTTP: 200
URL final: https://enriqueite.com/demoGPT/
Tipo de contenido: text/html
Servidor: HTTPd
Meta generator: no encontrado

Indicadores:
  No - wp-content en HTML
  No - wp-includes en HTML
  No - REST API indicada en cabecera
  No - REST API indicada en HTML
  No - wp-json mencionado en HTML

Conclusión: WordPress no confirmado.

=== COMPROBACIÓN DE RUTAS ===
HTTP   TAMAÑO     RUTA                                   REDIRECCIÓN
----------------------------------------------------------------------------------------------------
Referencia inexistente: 404 | tamaño declarado: ?
404    ?          wp-admin/                               (comparar con referencia)
404    ?          wp-login.php                            (comparar con referencia)
404    ?          wp-content/                             (comparar con referencia)
404    ?          wp-includes/                            (comparar con referencia)
404    ?          wp-json/          

detectar la api-reset ya  que no devolvio directamente si es wordpres

In [3]:

import requests
import json

url = "https://enriqueite.com/demoGPT/?rest_route=/"

try:
    r = requests.get(
        url,
        headers={"User-Agent": "CyberwallAcademy-EducationalAudit/1.0"},
        timeout=10
    )

    print("Estado HTTP:", r.status_code)
    print("Content-Type:", r.headers.get("Content-Type"))
    print("URL final:", r.url)

    if r.status_code == 429:
        print("El servidor limita las solicitudes. Detené las pruebas.")
    elif r.status_code == 200:
        contenido = r.text[:2500]
        try:
            datos = r.json()
            if isinstance(datos, dict):
                # Mostrar únicamente nombres de campos en el primer nivel.
                print("Tipo de respuesta JSON:", type(datos).__name__)
                print("Campos:", list(datos.keys())[:30])
                print("Vista parcial:")
                print(json.dumps(datos, indent=2, ensure_ascii=False)[:1500])
            else:
                print("JSON recibido; tipo:", type(datos).__name__)
                print("Vista parcial:", repr(datos)[:1000])
        except ValueError:
            print("La respuesta no es JSON válido.")
            print("Vista parcial:", contenido[:1000])
    else:
        print("Respuesta distinta de 200; no se continúa.")

except requests.RequestException as e:
    print("Error de conexión:", e)


Estado HTTP: 200
Content-Type: text/html
URL final: https://enriqueite.com/demoGPT/?rest_route=/
La respuesta no es JSON válido.
Vista parcial: <!DOCTYPE html>
<html>
  <head>
    <title>H4ck3d</title>
    <style>
      body {
        background-color: black;
        overflow: hidden;
        margin: 0;
      }
      canvas {
        display: block;
        position: fixed;
        top: 0;
        left: 0;
        z-index: -1;
      }
      .content {
        display: flex;
        flex-direction: column;
        justify-content: center;
        align-items: center;
        height: 100vh;
        text-align: center;
        font-family: "Courier New", monospace;
      }
      h1 {
        color: lime;
        text-shadow: 0px 0px 20px lime;
        font-size: 3em;
        margin: 0;
      }
      p {
        color: gray;
        text-shadow: 0px 0px 10px gray;
        font-size: 1.5em;
        margin-top: 10px;
      }
      .cursor {
        display: inline-block;
        background-c

examinar portada de la respuesta encontrada devuelta por el codigo anterior

In [4]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://enriqueite.com/demoGPT/"

r = requests.get(
    BASE,
    headers={"User-Agent": "CyberwallAcademy-EducationalAudit/1.0"},
    timeout=10
)

print("Estado:", r.status_code)
print("Tipo:", r.headers.get("Content-Type"))
print("URL final:", r.url)

soup = BeautifulSoup(r.text, "html.parser")

print("\n=== METADATOS ===")
print("Título:", soup.title.get_text(strip=True) if soup.title else "No encontrado")

for meta in soup.find_all("meta"):
    name = meta.get("name") or meta.get("property")
    content = meta.get("content")
    if name and content:
        print(f"{name}: {content[:200]}")

print("\n=== RECURSOS REFERENCIADOS ===")
for tag, attr in [("script", "src"), ("link", "href")]:
    for elemento in soup.find_all(tag):
        valor = elemento.get(attr)
        if valor:
            print(urljoin(r.url, valor))

print("\n=== INDICIOS EN EL HTML ===")
html = r.text.lower()

for indicador in [
    "wp-content", "wp-includes", "wp-json",
    "wordpress", "jquery", "generator",
    "h4ck3d", "hacked"
]:
    print(f"{indicador}: {'Sí' if indicador in html else 'No'}")


Estado: 200
Tipo: text/html
URL final: https://enriqueite.com/demoGPT/

=== METADATOS ===
Título: H4ck3d

=== RECURSOS REFERENCIADOS ===

=== INDICIOS EN EL HTML ===
wp-content: No
wp-includes: No
wp-json: No
wordpress: No
jquery: No
generator: No
h4ck3d: Sí
hacked: No


Este código analiza el HTML que entrega la portada y resume su estructura

In [5]:

import requests
from bs4 import BeautifulSoup
from collections import Counter

url = "https://enriqueite.com/demoGPT/"

r = requests.get(url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

print("=== ESTRUCTURA HTML ===")
print("Estado:", r.status_code)
print("Tamaño del HTML:", len(r.content), "bytes")
print("Etiquetas:", dict(Counter(
    tag.name for tag in soup.find_all(True)
)))

print("\n=== FORMULARIOS ===")
formularios = soup.find_all("form")
print("Cantidad:", len(formularios))

for i, form in enumerate(formularios, 1):
    print(
        i,
        "método =", form.get("method", "GET"),
        "| destino =", form.get("action", "(página actual)")
    )

print("\n=== ENLACES ===")
for a in soup.find_all("a", href=True):
    print(a.get_text(" ", strip=True)[:80], "->", a["href"])

print("\n=== COMENTARIOS HTML ===")
from bs4 import Comment

comentarios = soup.find_all(
    string=lambda text: isinstance(text, Comment)
)

print("Cantidad:", len(comentarios))
for comentario in comentarios[:10]:
    print(str(comentario).strip()[:200])


=== ESTRUCTURA HTML ===
Estado: 200
Tamaño del HTML: 3994 bytes
Etiquetas: {'html': 1, 'head': 1, 'title': 1, 'style': 1, 'body': 1, 'canvas': 1, 'div': 1, 'h1': 1, 'p': 1, 'script': 1}

=== FORMULARIOS ===
Cantidad: 0

=== ENLACES ===

=== COMENTARIOS HTML ===
Cantidad: 0


Este código no vuelve a consultar el sitio: analiza el HTML guardado en la variable r del último notebook, si todavía existe

In [6]:

from bs4 import BeautifulSoup
import hashlib
import re

# Reutiliza la respuesta r obtenida anteriormente.
html = r.text
soup = BeautifulSoup(html, "html.parser")

print("=== ANÁLISIS LOCAL DEL HTML ===")
print("SHA-256:", hashlib.sha256(r.content).hexdigest())
print("Tamaño:", len(r.content), "bytes")

# Scripts y recursos
scripts = soup.find_all("script")
print("\nScripts:", len(scripts))

for i, script in enumerate(scripts, 1):
    if script.get("src"):
        print(f"Script {i}: recurso externo")
    else:
        codigo = script.get_text()
        print(f"Script {i}: código integrado, {len(codigo)} caracteres")

# Formularios y elementos interactivos
print("\nFormularios:", len(soup.find_all("form")))
print("Enlaces:", len(soup.find_all("a")))
print("Canvas:", len(soup.find_all("canvas")))

# Indicadores tecnológicos encontrados literalmente
patrones = {
    "WordPress": r"wordpress|wp-content|wp-includes",
    "PHP": r"\.php(?:[?#\"'])",
    "API REST": r"wp-json|api/",
    "Frameworks JS": r"react|vue|angular|next\.js",
}

print("\n=== INDICADORES EN EL HTML ===")
for nombre, patron in patrones.items():
    coincidencias = re.findall(patron, html, re.IGNORECASE)
    print(f"{nombre}: {len(coincidencias)} coincidencias")


=== ANÁLISIS LOCAL DEL HTML ===
SHA-256: aaa36614a211099f5cedfc7578b2dddf6e9a6bea5d15befbeee314013ec0cc51
Tamaño: 3994 bytes

Scripts: 1
Script 1: código integrado, 2682 caracteres

Formularios: 0
Enlaces: 0
Canvas: 1

=== INDICADORES EN EL HTML ===
WordPress: 0 coincidencias
PHP: 0 coincidencias
API REST: 0 coincidencias
Frameworks JS: 0 coincidencias


inspeccionar las cabeceras HTTP

In [7]:

import requests

url = "https://enriqueite.com/demoGPT/"

try:
    respuesta = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; EducationalAudit/1.0)"},
        timeout=10
    )

    print("=== CABECERAS HTTP ===")
    print("Estado:", respuesta.status_code)

    relevantes = [
        "Server",
        "X-Powered-By",
        "Content-Type",
        "Content-Length",
        "Location",
        "Link",
        "Allow",
        "Cache-Control",
        "Via",
        "X-Redirect-By",
    ]

    for nombre in relevantes:
        valor = respuesta.headers.get(nombre)
        if valor:
            print(f"{nombre}: {valor}")

    print("\n=== COOKIES ===")
    if respuesta.cookies:
        for cookie in respuesta.cookies:
            print(f"{cookie.name}: presente (valor oculto)")
    else:
        print("No se recibieron cookies.")

except requests.RequestException as error:
    print("Error de conexión:", error)


=== CABECERAS HTTP ===
Estado: 200
Server: HTTPd
Content-Type: text/html
Content-Length: 1419

=== COOKIES ===
No se recibieron cookies.


In [8]:

from bs4 import BeautifulSoup
import hashlib

print("=== COMPARACIÓN DE LA RESPUESTA ACTUAL ===")

print("Estado HTTP:", respuesta.status_code)
print("Content-Length declarado:",
      respuesta.headers.get("Content-Length", "No indicado"))
print("Tamaño real recibido:", len(respuesta.content), "bytes")
print("SHA-256:",
      hashlib.sha256(respuesta.content).hexdigest())

soup = BeautifulSoup(respuesta.text, "html.parser")

print("Título:",
      soup.title.get_text(strip=True) if soup.title else "Sin título")

print("\nTexto visible:")
print(soup.get_text(" ", strip=True)[:500])

print("\nCantidad de scripts:", len(soup.find_all("script")))
print("Cantidad de enlaces:", len(soup.find_all("a")))


=== COMPARACIÓN DE LA RESPUESTA ACTUAL ===
Estado HTTP: 200
Content-Length declarado: 1419
Tamaño real recibido: 3994 bytes
SHA-256: aaa36614a211099f5cedfc7578b2dddf6e9a6bea5d15befbeee314013ec0cc51
Título: H4ck3d

Texto visible:
H4ck3d

Cantidad de scripts: 1
Cantidad de enlaces: 0


comprobar el protocolo http

In [9]:

print("=== DATOS DE TRANSPORTE ===")

for nombre in [
    "Content-Length",
    "Content-Encoding",
    "Transfer-Encoding",
    "Connection",
    "Date",
    "ETag",
    "Last-Modified",
    "Vary",
]:
    print(f"{nombre}: {respuesta.headers.get(nombre, 'No indicado')}")

print("\n=== DATOS DE LA SESIÓN ===")
print("Protocolo HTTP:", respuesta.raw.version)
print("Bytes recibidos por requests:", len(respuesta.content))
print("Bytes del texto codificado UTF-8:",
      len(respuesta.text.encode("utf-8")))


=== DATOS DE TRANSPORTE ===
Content-Length: 1419
Content-Encoding: gzip
Transfer-Encoding: No indicado
Connection: keep-alive
Date: Thu, 29 Oct 2026 14:19:40 GMT
ETag: "f9a-642145596f6c0-gzip"
Last-Modified: Sun, 26 Oct 2025 18:56:03 GMT
Vary: Accept-Encoding

=== DATOS DE LA SESIÓN ===
Protocolo HTTP: 11
Bytes recibidos por requests: 3994
Bytes del texto codificado UTF-8: 3994


In [10]:
import requests
import hashlib

url = "https://enriqueite.com/demoGPT/"

respuesta = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0 (compatible; curso-auditoria/1.0)",
        "Accept-Encoding": "identity"
    },
    timeout=15
)

print("Estado HTTP:", respuesta.status_code)
print("Content-Encoding:", respuesta.headers.get("Content-Encoding", "No indicado"))
print("Content-Length:", respuesta.headers.get("Content-Length", "No indicado"))
print("Bytes recibidos:", len(respuesta.content))
print("SHA-256:", hashlib.sha256(respuesta.content).hexdigest())
print("Título:", respuesta.text[:500])

Estado HTTP: 200
Content-Encoding: No indicado
Content-Length: No indicado
Bytes recibidos: 3994
SHA-256: aaa36614a211099f5cedfc7578b2dddf6e9a6bea5d15befbeee314013ec0cc51
Título: <!DOCTYPE html>
<html>
  <head>
    <title>H4ck3d</title>
    <style>
      body {
        background-color: black;
        overflow: hidden;
        margin: 0;
      }
      canvas {
        display: block;
        position: fixed;
        top: 0;
        left: 0;
        z-index: -1;
      }
      .content {
        display: flex;
        flex-direction: column;
        justify-content: center;
        align-items: center;
        height: 100vh;
        text-align: center;
        font-family:
